# BigQuant single-factor submission

This notebook is generated for exactly one factor submission. Upload this `.ipynb` file alone.


In [ ]:
def main(datasources, start_date, end_date):
    """Return one AutoMiner v2 intraday temporal-shape factor."""

    import json
    import numpy as np
    import pandas as pd
    import dai

    cfg = json.loads('{"expected_failure_mode": "trade-size bridge is dominated by size exposure", "factor_id": "amv2_prev_trade_size_late_bridge_e0b7a6c3", "mechanism_family": "PREV_TRADE_SIZE_LATE_BRIDGE", "meta_pattern": "sequence", "signal_sources": ["bar1m.prev_avg_trade_size", "bar1m.late_avg_trade_size", "bar1m.early_avg_trade_size", "exposure.SIZE"]}')
    family = cfg["mechanism_family"]

    def _source(keys, fallback):
        if isinstance(datasources, dict):
            for key in keys:
                if key in datasources and datasources[key] is not None:
                    return datasources[key]
        return fallback

    bar1m = _source(["bar1m", "stock_bar1m", "bigalpha_2026_stock_bar1m"], "bigalpha_2026_stock_bar1m")
    instruments = _source(["instruments", "instrument", "bigalpha_2026_instruments"], "bigalpha_2026_instruments")
    exposure = _source(["exposure", "bigalpha_2026_exposure"], "bigalpha_2026_exposure")

    start_ts = pd.Timestamp(start_date).normalize()
    end_exclusive = pd.Timestamp(end_date).normalize() + pd.DateOffset(days=1)
    query_left = (start_ts - pd.DateOffset(days=10)).strftime("%Y-%m-%d")
    query_right = end_exclusive.strftime("%Y-%m-%d")


    def _month_windows(start_value, end_value):
        cursor = pd.Timestamp(year=start_value.year, month=start_value.month, day=1)
        if cursor < start_value:
            cursor = start_value
        while cursor < end_value:
            next_month = pd.Timestamp(year=cursor.year, month=cursor.month, day=1) + pd.DateOffset(months=1)
            right = min(next_month.normalize(), end_value)
            if right <= cursor:
                break
            yield cursor.strftime("%Y-%m-%d"), right.strftime("%Y-%m-%d")
            cursor = right

    def _safe_num(frame, column, default=np.nan):
        if column not in frame.columns:
            return pd.Series(default, index=frame.index, dtype=float)
        return pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

    def _rank_by_date(frame, series):
        values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
        return values.groupby(frame["date"], sort=False).rank(pct=True).fillna(0.5)

    def _query_universe(left, right):
        try:
            frame = dai.query(
                "SELECT date, instrument FROM " + str(instruments),
                filters={"date": [left, right]},
                compression=True,
            ).df()
        except Exception:
            return pd.DataFrame(columns=["date", "instrument"])
        if frame.empty:
            return pd.DataFrame(columns=["date", "instrument"])
        frame = frame.copy()
        frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        frame = frame.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"])
        return frame[(frame["date"] >= start_ts) & (frame["date"] < end_exclusive)]

    def _query_bar(left, right):
        fields = ["date", "instrument", "open", "high", "low", "close", "amount", "volume", "deal_number"]
        ohlcv_fields = ["date", "instrument", "open", "high", "low", "close", "amount", "volume"]
        minimal_fields = ["date", "instrument", "open", "high", "low", "close"]
        for selected in (fields, ohlcv_fields, minimal_fields):
            try:
                frame = dai.query(
                    "SELECT " + ", ".join(selected) + " FROM " + str(bar1m),
                    filters={"date": [left, right]},
                    compression=True,
                ).df()
                if not frame.empty:
                    return frame
            except Exception:
                pass
        return pd.DataFrame(columns=fields)

    def _query_exposure(left, right):
        fields = ["date", "instrument", "RESVOL", "LIQUIDTY", "BTOP", "SIZE"]
        try:
            frame = dai.query(
                "SELECT " + ", ".join(fields) + " FROM " + str(exposure),
                filters={"date": [left, right]},
                compression=True,
            ).df()
        except Exception:
            return pd.DataFrame(columns=fields)
        if frame.empty:
            return pd.DataFrame(columns=fields)
        frame = frame.copy()
        frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        for column in ["RESVOL", "LIQUIDTY", "BTOP", "SIZE"]:
            frame[column] = _safe_num(frame, column)
        return frame.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"], keep="last")

    def _aggregate_bar(raw):
        if raw.empty:
            return pd.DataFrame(columns=["date", "instrument"])
        data = raw.copy()
        data["timestamp"] = pd.to_datetime(data["date"], errors="coerce")
        data["date"] = data["timestamp"].dt.normalize()
        data["instrument"] = data["instrument"].astype(str)
        data = data.dropna(subset=["timestamp", "date", "instrument"])
        if data.empty:
            return pd.DataFrame(columns=["date", "instrument"])
        for column in ["open", "high", "low", "close", "amount", "volume", "deal_number"]:
            data[column] = _safe_num(data, column, 0.0).fillna(0.0)
        minute_of_day = data["timestamp"].dt.hour * 60 + data["timestamp"].dt.minute
        data["bucket"] = np.where(minute_of_day <= 630, "early", np.where(minute_of_day >= 840, "late", "mid"))

        grouped = data.groupby(["date", "instrument"], sort=False)
        daily = grouped.agg(
            amount_sum=("amount", "sum"),
            volume_sum=("volume", "sum"),
            deal_sum=("deal_number", "sum"),
            open_first=("open", "first"),
            close_last=("close", "last"),
            high_max=("high", "max"),
            low_min=("low", "min"),
        ).reset_index()

        def _bucket(prefix, name):
            subset = data[data["bucket"] == name]
            if subset.empty:
                return daily[["date", "instrument"]].assign(**{
                    prefix + "_amount": 0.0,
                    prefix + "_deal": 0.0,
                    prefix + "_open": np.nan,
                    prefix + "_close": np.nan,
                })
            out = subset.groupby(["date", "instrument"], sort=False).agg(
                **{
                    prefix + "_amount": ("amount", "sum"),
                    prefix + "_deal": ("deal_number", "sum"),
                    prefix + "_open": ("open", "first"),
                    prefix + "_close": ("close", "last"),
                }
            ).reset_index()
            return out

        for prefix, name in [("early", "early"), ("mid", "mid"), ("late", "late")]:
            daily = daily.merge(_bucket(prefix, name), on=["date", "instrument"], how="left")

        for column in [
            "early_amount",
            "mid_amount",
            "late_amount",
            "early_deal",
            "mid_deal",
            "late_deal",
        ]:
            daily[column] = pd.to_numeric(daily.get(column, 0.0), errors="coerce").fillna(0.0)

        denom = daily["open_first"].where(daily["open_first"].abs() > 1e-12, np.nan)
        daily["ret"] = ((daily["close_last"] - daily["open_first"]) / denom).replace([np.inf, -np.inf], np.nan)
        daily["path_range"] = ((daily["high_max"] - daily["low_min"]) / denom.abs()).replace([np.inf, -np.inf], np.nan)
        daily["early_ret"] = ((daily["early_close"] - daily["open_first"]) / denom).replace([np.inf, -np.inf], np.nan)
        late_open = daily["late_open"].where(daily["late_open"].abs() > 1e-12, np.nan)
        daily["late_ret"] = ((daily["close_last"] - late_open) / late_open).replace([np.inf, -np.inf], np.nan)
        daily["range_efficiency"] = (daily["ret"].abs() / daily["path_range"].where(daily["path_range"].abs() > 1e-12, np.nan)).replace([np.inf, -np.inf], np.nan)
        daily["avg_trade_size"] = (daily["amount_sum"] / daily["deal_sum"].where(daily["deal_sum"] > 0, np.nan)).replace([np.inf, -np.inf], np.nan)
        daily["early_avg_trade_size"] = (daily["early_amount"] / daily["early_deal"].where(daily["early_deal"] > 0, np.nan)).replace([np.inf, -np.inf], np.nan)
        daily["late_avg_trade_size"] = (daily["late_amount"] / daily["late_deal"].where(daily["late_deal"] > 0, np.nan)).replace([np.inf, -np.inf], np.nan)
        amount_denom = daily["amount_sum"].where(daily["amount_sum"] > 0, np.nan)
        for prefix in ["early", "mid", "late"]:
            daily[prefix + "_amount_share"] = (daily[prefix + "_amount"] / amount_denom).replace([np.inf, -np.inf], np.nan).fillna(0.0)

        shares = daily[["early_amount_share", "mid_amount_share", "late_amount_share"]].clip(lower=0.0)
        entropy = -(shares.where(shares > 0, 1.0).apply(np.log) * shares).sum(axis=1) / np.log(3.0)
        daily["volume_entropy"] = entropy.replace([np.inf, -np.inf], np.nan)
        daily = daily.sort_values(["instrument", "date"])
        daily["prev_close"] = daily.groupby("instrument", sort=False)["close_last"].shift(1)
        daily["open_gap"] = ((daily["open_first"] - daily["prev_close"]) / daily["prev_close"].where(daily["prev_close"].abs() > 1e-12, np.nan)).replace([np.inf, -np.inf], np.nan)
        return daily

    chunks = []
    for left, right in _month_windows(start_ts, end_exclusive):
        history_left = (pd.Timestamp(left) - pd.DateOffset(days=15)).strftime("%Y-%m-%d")
        universe = _query_universe(left, right)
        if universe.empty:
            continue

        daily_all = _aggregate_bar(_query_bar(history_left, right))
        if daily_all.empty:
            panel = universe.copy()
        else:
            panel = daily_all.merge(universe[["date", "instrument"]], on=["date", "instrument"], how="outer")

        exp = _query_exposure(left, right)
        panel = panel.merge(exp, on=["date", "instrument"], how="left") if not exp.empty else panel

        for column in [
            "amount_sum",
            "deal_sum",
            "ret",
            "path_range",
            "early_ret",
            "late_ret",
            "range_efficiency",
            "avg_trade_size",
            "early_avg_trade_size",
            "late_avg_trade_size",
            "early_amount_share",
            "mid_amount_share",
            "late_amount_share",
            "volume_entropy",
            "open_gap",
            "RESVOL",
            "LIQUIDTY",
            "BTOP",
            "SIZE",
        ]:
            if column not in panel.columns:
                panel[column] = np.nan
            panel[column] = pd.to_numeric(panel[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

        panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
        panel["instrument"] = panel["instrument"].astype(str)
        panel = panel.dropna(subset=["date", "instrument"]).sort_values(["instrument", "date"])
        for source_column, lag_column in [
            ("ret", "prev_ret"),
            ("path_range", "prev_path_range"),
            ("amount_sum", "prev_amount_sum"),
            ("volume_entropy", "prev_volume_entropy"),
            ("avg_trade_size", "prev_avg_trade_size"),
        ]:
            panel[lag_column] = panel.groupby("instrument", sort=False)[source_column].shift(1)
        panel = panel.loc[(panel["date"] >= start_ts) & (panel["date"] < end_exclusive)].copy()
        if panel.empty:
            continue

        ret_rank = _rank_by_date(panel, panel["ret"])
        early_ret_rank = _rank_by_date(panel, panel["early_ret"])
        late_ret_rank = _rank_by_date(panel, panel["late_ret"])
        gap_abs_rank = _rank_by_date(panel, panel["open_gap"].abs())
        gap_sign = np.sign(panel["open_gap"]).replace(0, np.nan).fillna(1.0)
        early_share = _rank_by_date(panel, panel["early_amount_share"])
        late_share = _rank_by_date(panel, panel["late_amount_share"])
        path_range = _rank_by_date(panel, panel["path_range"])
        late_size = _rank_by_date(panel, panel["late_avg_trade_size"])
        early_size = _rank_by_date(panel, panel["early_avg_trade_size"])
        prev_ret_rank = _rank_by_date(panel, panel["prev_ret"].where(panel["prev_ret"].notna(), panel["ret"]))
        prev_range_rank = _rank_by_date(panel, panel["prev_path_range"].where(panel["prev_path_range"].notna(), panel["path_range"]))
        prev_amount_rank = _rank_by_date(panel, panel["prev_amount_sum"].where(panel["prev_amount_sum"].notna(), panel["amount_sum"]))
        prev_entropy_rank = _rank_by_date(panel, panel["prev_volume_entropy"].where(panel["prev_volume_entropy"].notna(), panel["volume_entropy"]))
        prev_size_rank = _rank_by_date(panel, panel["prev_avg_trade_size"].where(panel["prev_avg_trade_size"].notna(), panel["avg_trade_size"]))
        resvol = _rank_by_date(panel, panel["RESVOL"].where(panel["RESVOL"].notna(), panel["path_range"]))
        liq = _rank_by_date(panel, panel["LIQUIDTY"].where(panel["LIQUIDTY"].notna(), panel["amount_sum"]))
        size = _rank_by_date(panel, panel["SIZE"].where(panel["SIZE"].notna(), panel["amount_sum"]))
        btop = _rank_by_date(panel, panel["BTOP"].where(panel["BTOP"].notna(), -panel["avg_trade_size"]))

        size_bridge = late_size - early_size
        raw = (prev_size_rank - 0.5) * size_bridge * (late_ret_rank - 0.5) - 0.15 * (size - 0.5)
        raw = pd.to_numeric(raw, errors="coerce").replace([np.inf, -np.inf], np.nan)
        if int(raw.nunique(dropna=True)) <= 1:
            raw = raw.fillna(0.0) + 0.01 * (late_ret_rank - 0.5) + 0.005 * (prev_ret_rank - 0.5)
        panel["factor"] = _rank_by_date(panel, raw).replace([np.inf, -np.inf], np.nan).fillna(0.5)
        chunks.append(panel[["date", "instrument", "factor"]].copy())

    if chunks:
        out = pd.concat(chunks, ignore_index=True).drop_duplicates(["date", "instrument"], keep="last")
    else:
        out = pd.DataFrame(columns=["date", "instrument", "factor"])
    if out.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    out["instrument"] = out["instrument"].astype(str)
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.5)
    return out[["date", "instrument", "factor"]].dropna(subset=["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
